In [1]:
import numpy
import torchvision

DATASET = "E:/DiplomaV2/full_run_2/mnist/results"

train = torchvision.datasets.MNIST('mnist', train = True, download = False)
test = torchvision.datasets.MNIST('mnist', train = False, download = False)

train_images = numpy.array([numpy.array(item[0]) / 255 for item in train])
train_labels = numpy.array([item[1] for item in train])

test_images = numpy.array([numpy.array(item[0]) / 255 for item in test])
test_labels = numpy.array([item[1] for item in test])

In [2]:
import cvtda.topology

fe = cvtda.topology.FeatureExtractor(n_jobs = 1)
train_features = fe.fit_transform(train_images, dump_name = f'{DATASET}/train')
test_features = fe.transform(test_images, dump_name = f'{DATASET}/test')

GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/mnist/results/train/greyscale, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/mnist/results/train/greyscale/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/mnist/results/train/greyscale/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/mnist/results/train/inverted_greyscale, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/mnist/results/train/inverted_greyscale/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/mnist/results/train/inverted_greyscale/features.npy
Fitting filtrations


100%|██████████| 72/72 [02:09<00:00,  1.80s/it]


Got the result from E:/DiplomaV2/full_run_2/mnist/results/train/geometry/features.npy
Applying StandardScaler.
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/mnist/results/test/greyscale, do_fit = False
Got the result from E:/DiplomaV2/full_run_2/mnist/results/test/greyscale/diagrams.npy
Applying Scaler to persistence diagrams.
Got the result from E:/DiplomaV2/full_run_2/mnist/results/test/greyscale/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/mnist/results/test/inverted_greyscale, do_fit = False
Got the result from E:/DiplomaV2/full_run_2/mnist/results/test/inverted_greyscale/diagrams.npy
Applying Scaler to persistence diagrams.
Got the result from E:/DiplomaV2/full_run_2/mnist/results/test/inverted_greyscale/features.npy
Applying filtrations


100%|██████████| 72/72 [00:13<00:00,  5.27it/s]


Got the result from E:/DiplomaV2/full_run_2/mnist/results/test/geometry/features.npy
Applying StandardScaler.


In [3]:
import cvtda.utils
import sklearn.tree
import cvtda.topology
import sklearn.metrics
import cvtda.classification

def explain_decision_tree(fe: cvtda.topology.FeatureExtractor, dt: sklearn.tree.DecisionTreeClassifier, obj: int):
    target, prediction = test_labels[obj], dt.predict([test_features[obj]])[0]
    match = "✓" if target == prediction else "✗"

    path = dt.decision_path([test_features[obj]]).toarray()[0]
    features_idxs = dt.tree_.feature[numpy.where(path != 0)[0][:-1]]
    feature_names = list(numpy.array(fe.feature_names())[features_idxs])
    return cvtda.utils.FeatureExplanation.display_many(
        [fe.explain(f, test_images[obj]) for f in feature_names],
        title=f"Object {obj}  ·  target: {target}   prediction: {prediction} {match}",
    )

dt = sklearn.tree.DecisionTreeClassifier(max_depth = 8, max_features = 1/3, random_state = 42)
dt.fit(train_features, train_labels)
cvtda.classification.estimate_quality(dt.predict_proba(test_features), test_labels)

{'Accuracy': 0.8896,
 'AUC-ROC': 0.9839846157440252,
 'Precision': 0.8960454403928072,
 'Recall': 0.8878350623301656,
 'F1-score': 0.8898620070540213,
 'TOP-2 Accuracy': 0.9277,
 'TOP-3 Accuracy': 0.9507,
 'TOP-5 Accuracy': 0.9793,
 'TOP-7 Accuracy': 0.9927,
 'TOP-9 Accuracy': 0.999}

In [4]:
import os
import tqdm
import joblib
import matplotlib.pyplot as plt

def process_item(i):
    plt.rcParams.update({ 'font.size': 13 })
    fig = explain_decision_tree(fe, dt, i)
    y = test_labels[i]
    os.makedirs(f"cvtda_results/mnist/{y}", exist_ok=True)
    fig.savefig(f"cvtda_results/mnist/{y}/{i}.png")
    fig.savefig(f"cvtda_results/mnist/{y}/{i}.svg")
    plt.close(fig)

joblib.Parallel(n_jobs=-1)(joblib.delayed(process_item)(i) for i in tqdm.trange(len(test)))
None

100%|██████████| 10000/10000 [25:43<00:00,  6.48it/s]
